# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARTI-INTEL/fr-ml-intern-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring**

This contract says exactly what one row means, which fields are features vs labels, and which time windows each covers. Every claim below has a query cell next to it.

---
## Setup — connect to the warehouse

In [1]:
import duckdb, os, getpass, warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

warnings.filterwarnings('ignore')

# Hugging Face token — never paste into code, use env var or getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print('Connected to warehouse')

Connected to warehouse


---
## 1. The contract — in plain words

### 1a. What one row means
**One row = one content item (one page)** assessed at the decision moment of **2026-03-31** (end of the mid-panel month). The row answers: *should this page be reviewed for refresh?*

### 1b. Which table(s)
| Table | Role |
|---|---|
| `fact_content_daily_performance` (partitions 2026-01 through 2026-03) | Daily metrics: impressions, clicks, position for feature and label windows |
| `dim_content` | Fixed content attributes: word count, content age, intent |

### 1c. Time windows
```
  Feature window (Feb 2026)         Label window (Mar 2026)
  ├─────────────────────────└        ├────────────────────────────┘
  2026-02-01 ... 2026-02-28         2026-03-01 ... 2026-03-31
       
                                ▲
                          Decision moment
                          (end of March)
```
- **Feature window**: 2026-02-01 to 2026-02-28 (28 days) — metrics fully known before the decision moment
- **Label window**: 2026-03-01 to 2026-03-31 (31 days) — the outcome period we predict

### 1d. What we predict (label / proxy)
**`is_declining`** — a binary proxy. 1 if the page's total impressions in the label window (March) were **less than 80%** of its impressions in the feature window (February). That means a ≥20% month-over-month drop.

> This is a **proxy** label — it measures whether a decline *has already happened*, not whether a refresh *will help*. A future-looking label (will this page decline next month?) is the upgrade path for the capstone.

### 1e. One thing deliberately excluded
**`fact_content_query_90d`** — its fixed 90-day window covers December 2025 through February 2026, which **overlaps** the feature window. Per-content aggregates like `content_total_impressions_90d` would include the feature window itself, making train/test separation ambiguous. We exclude it to keep a clean temporal boundary.

---
## 2. Prove it with queries (mid-panel month = 2026-03)

Three facts, three queries.

### Fact 1 — The grain holds

**Claim:** One row in `fact_content_daily_performance` = one `report_date` × one `client_hash_id` × one `content_hash_id`. No duplicate combinations exist within month=2026-03.

In [2]:
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

if len(dupes) == 0:
    print("✓ Grain holds: zero duplicate (date, client, content) combinations in month=2026-03")
else:
    print(f"✗ Grain violated — {len(dupes)} duplicates found")
    print(dupes)

✓ Grain holds: zero duplicate (date, client, content) combinations in month=2026-03


### Fact 2 — Row count and date span

**Claim:** The March 2026 partition contains daily data spanning 2026-03-01 through 2026-03-31.

In [3]:
row_count = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

rc = row_count['row_count'].iloc[0]
fd = row_count['first_date'].iloc[0]
ld = row_count['last_date'].iloc[0]
days = (ld - fd).days + 1
print(f"Rows in month=2026-03: {rc:,}")
print(f"Date span:            {fd} to {ld} ({days} days)")

Rows in month=2026-03: 9,841,378
Date span:            2026-03-01 00:00:00 to 2026-03-31 00:00:00 (31 days)


### Fact 3 — Availability filter with `IS TRUE`

**Claim:** Filtering with `ga4_data_available IS TRUE` removes rows where GA4 columns are zero-filled placeholders. Most rows survive because March 2026 is within most clients' GA4 history.

In [4]:
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_avail_rows,
           ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS ga4_avail_pct,
           SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_not_avail_rows,
           SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS ga4_null_rows
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

print(f"Total rows in March 2026:            {avail['total_rows'].iloc[0]:,}")
print(f"Rows with ga4_data_available IS TRUE:  {avail['ga4_avail_rows'].iloc[0]:,}  ({avail['ga4_avail_pct'].iloc[0]}%)")
print(f"Rows with ga4_data_available NOT TRUE: {avail['ga4_not_avail_rows'].iloc[0]:,}")
print(f"  (of which NULL:                       {avail['ga4_null_rows'].iloc[0]:,})")

Total rows in March 2026:            9,841,378
Rows with ga4_data_available IS TRUE:  413,966.0  (4.2%)
Rows with ga4_data_available NOT TRUE: 9,427,412.0
  (of which NULL:                       3,018,741.0)


---
## 3. Five features (max) — built from the same mid-panel month

Features are aggregated per content item from the **feature window (February 2026)**. Each is labeled as knowable at the decision moment because the measurement was completed before March (the label window) began.

In [5]:
# Build the feature frame: aggregate per content across Feb (feature window) and Mar (label window)
feature_frame = con.sql(f"""
    WITH feb_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions)  AS imp_feb,       -- Feature 1: total impressions in Feb
            AVG(gsc_avg_position) AS pos_feb,        -- Feature 2: avg position in Feb
            SUM(gsc_clicks)       AS clk_feb,        -- Feature 3: total clicks in Feb
            COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_imp_feb  -- Feature 4
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
        GROUP BY client_hash_id, content_hash_id
    ),
    mar_agg AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS imp_mar          -- for the label only
        FROM {TABLES['fact_daily']}
        WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
        GROUP BY content_hash_id
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.imp_feb,
        f.pos_feb,
        f.clk_feb,
        f.days_with_imp_feb,
        m.imp_mar
    FROM feb_agg f
    JOIN mar_agg m USING (content_hash_id)
    WHERE f.imp_feb >= 100   -- minimum volume filter
""").df()

print(f"Feature frame: {len(feature_frame):,} content items with ≥100 impressions in February")
print(f"Columns: {list(feature_frame.columns)}")
feature_frame.head()

Feature frame: 76,837 content items with ≥100 impressions in February
Columns: ['client_hash_id', 'content_hash_id', 'imp_feb', 'pos_feb', 'clk_feb', 'days_with_imp_feb', 'imp_mar']


,client_hash_id,content_hash_id,imp_feb,pos_feb,clk_feb,days_with_imp_feb,imp_mar
0,client_9958f0a7ae1df715,content_3e04f9d434376e2d,166.0,22.165154,0.0,27,177.0
1,client_9958f0a7ae1df715,content_11f0897236c3049d,161.0,60.621348,0.0,27,195.0
2,client_9958f0a7ae1df715,content_c1d7772eada473df,459.0,30.142923,2.0,28,619.0
3,client_9958f0a7ae1df715,content_515a1d5cbb1d5442,1173.0,12.935718,1.0,28,1044.0
4,client_9958f0a7ae1df715,content_22e45517ce5365c5,119.0,7.754032,0.0,26,118.0


### Feature 1: `imp_feb` — total GSC impressions in February
**Knowable at the decision moment because** February ended before March began, so every impression was already counted before the label window started.

### Feature 2: `pos_feb` — average GSC position in February
**Knowable at the decision moment because** position data from February was fully recorded by end-of-month and available before the label period.

### Feature 3: `clk_feb` — total GSC clicks in February
**Knowable at the decision moment because** all February click events were measured before March 1st.

### Feature 4: `days_with_imp_feb` — distinct days with impressions in February
**Knowable at the decision moment because** the daily impression presence is a complete February measurement, finalized when the month ended.

### Feature 5: `content_age_days` — days since content creation (as of 2026-03-31)
**Knowable at the decision moment because** content's creation date is a fixed attribute from `dim_content.content_created_date`. Age is computed as `DATEDIFF('day', created_date, '2026-03-31')`, which is independent of the label window's performance data.

In [6]:
# Join dim_content to get Feature 5: compute content_age_days from content_created_date
content_meta = con.sql(f"""
    SELECT content_hash_id,
           DATEDIFF('day', content_created_date, DATE '2026-03-31') AS content_age_days
    FROM {TABLES['dim_content']}
    WHERE content_created_date IS NOT NULL
""").df()

feature_frame = feature_frame.merge(content_meta, on='content_hash_id', how='left')
feature_frame['content_age_days'] = feature_frame['content_age_days'].fillna(0)

# Define the label
feature_frame['is_declining'] = (feature_frame['imp_mar'] < 0.8 * feature_frame['imp_feb']).astype(int)

print(f"Label distribution:")
print(f"  Declining (label=1): {(feature_frame['is_declining'] == 1).sum():,} ({(feature_frame['is_declining'].mean()*100):.1f}%)")
print(f"  Stable/Up (label=0): {(feature_frame['is_declining'] == 0).sum():,} ({(1-feature_frame['is_declining'].mean())*100:.1f}%)")
print()
print("Final feature set (5 features) — all knowable before the label window:")
feature_cols = ['imp_feb', 'pos_feb', 'clk_feb', 'days_with_imp_feb', 'content_age_days']

preview = feature_frame[['content_hash_id'] + feature_cols + ['is_declining']].head(8).copy()
preview['content_hash_id'] = preview['content_hash_id'].str[:16] + '...'
preview

Label distribution:
  Declining (label=1): 13,989 (18.2%)
  Stable/Up (label=0): 62,848 (81.8%)

Final feature set (5 features) — all knowable before the label window:


,content_hash_id,imp_feb,pos_feb,clk_feb,days_with_imp_feb,content_age_days,is_declining
0,content_3e04f9d4...,166.0,22.165154,0.0,27,368,0
1,content_11f08972...,161.0,60.621348,0.0,27,368,0
2,content_c1d7772e...,459.0,30.142923,2.0,28,368,0
3,content_515a1d5c...,1173.0,12.935718,1.0,28,368,0
4,content_22e45517...,119.0,7.754032,0.0,26,368,0
5,content_ac03f12e...,136.0,40.805450,0.0,28,447,0
6,content_4f2fd80a...,208.0,11.530789,0.0,28,447,0
7,content_ab409515...,383.0,6.683083,0.0,28,447,0


---
## 4. The trap — add one label-derived column, watch the score jump, then delete it

This demonstrates the leakage lesson from notebook 02 on real warehouse data.

In [7]:
# --- HONEST MODEL (no leakage) ---
X_honest = feature_frame[feature_cols].copy()
y_honest = feature_frame['is_declining'].copy()

# Drop rows with missing feature values
valid = X_honest.notna().all(axis=1)
X_honest = X_honest[valid]
y_honest = y_honest[valid]

X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y_honest, test_size=0.25, random_state=42, stratify=y_honest)
model_honest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

honest_base = max(y_te.mean(), 1 - y_te.mean())
honest_auc = roc_auc_score(y_te, model_honest.predict_proba(X_te)[:, 1])

print("=" * 70)
print("HONEST MODEL — 5 features, no label leakage")
print("=" * 70)
print(f"Base rate (majority class): {honest_base:.3f}")
print(f"ROC AUC: {honest_auc:.3f}")
print()
print(classification_report(y_te, model_honest.predict(X_te), digits=3))

HONEST MODEL — 5 features, no label leakage
Base rate (majority class): 0.818
ROC AUC: 0.693



              precision    recall  f1-score   support

           0      0.834     0.969     0.897     15713
           1      0.491     0.136     0.213      3497

    accuracy                          0.817     19210
   macro avg      0.663     0.552     0.555     19210
weighted avg      0.772     0.817     0.772     19210



In [8]:
# --- TRAP MODEL (with label-derived column) ---
# imp_change_pct is the % change from Feb to March.
# This is LABEL-DERIVED because is_declining = (imp_mar < 0.8 * imp_feb)
# Knowing how much impressions changed directly tells you the label.

feature_frame['imp_change_pct'] = (feature_frame['imp_mar'] - feature_frame['imp_feb']) / feature_frame['imp_feb']

X_trap = feature_frame[feature_cols + ['imp_change_pct']].copy()
y_trap = feature_frame['is_declining'].copy()

valid = X_trap.notna().all(axis=1)
X_trap = X_trap[valid]
y_trap = y_trap[valid]

X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(X_trap, y_trap, test_size=0.25, random_state=42, stratify=y_trap)
model_trap = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_t, y_tr_t)

trap_auc = roc_auc_score(y_te_t, model_trap.predict_proba(X_te_t)[:, 1])

print("=" * 70)
print("TRAP MODEL — with imp_change_pct (label-derived column)")
print("=" * 70)
print(f"Base rate (majority class): {max(y_te_t.mean(), 1 - y_te_t.mean()):.3f}")
print(f"ROC AUC: {trap_auc:.4f}")
print()
print(classification_report(y_te_t, model_trap.predict(X_te_t), digits=3))

TRAP MODEL — with imp_change_pct (label-derived column)
Base rate (majority class): 0.818
ROC AUC: 1.0000

              precision    recall  f1-score   support

           0      1.000     1.000     1.000     15713
           1      1.000     1.000     1.000      3497

    accuracy                          1.000     19210
   macro avg      1.000     1.000     1.000     19210
weighted avg      1.000     1.000     1.000     19210



In [9]:
# --- DELETE the trap column, keep the honest number ---
if 'imp_change_pct' in feature_frame.columns:
    del feature_frame['imp_change_pct']
    print("✓ Deleted imp_change_pct — leakage column removed")

print()
print("=" * 70)
print("THE LEAKAGE LESSON — from notebook 02, verified on warehouse data")
print("=" * 70)
print()
print(f"  {'Model':<40} {'ROC AUC':>10} {'Note':<20}")
print(f"  {'-'*40} {'-'*10} {'-'*20}")
print(f"  {'Honest (5 features, no leakage)':<40} {honest_auc:>10.3f} {'Real earned signal':<20}")
print(f"  {'Trap (with imp_change_pct)':<40} {trap_auc:>10.3f} {'Near-perfect (leaked!)':<20}")
print(f"  {'Base rate (predict majority)':<40} {honest_base:>10.3f} {'Dumb baseline':<20}")
print()
print(f"What happened: imp_change_pct is the percent change from February impressions to")
print(f"March impressions. The label is_declining is defined as March being <80% of")
print(f"February. A model that sees imp_change_pct learns: if imp_change_pct is less")
print(f"than roughly -0.2, predict declining. That's DIRECT LEAKAGE — the feature")
print(f"contains the label's computation.")
print()
print(f"Lesson: Never feed a column derived from the same period you're trying to")
print(f"predict. trend_pct, impression_change_pct, or any rate that compares the")
print(f"label window to the feature window is leakage.")

✓ Deleted imp_change_pct — leakage column removed

THE LEAKAGE LESSON — from notebook 02, verified on warehouse data

  Model                                       ROC AUC Note                
  ---------------------------------------- ---------- --------------------
  Honest (5 features, no leakage)               0.693 Real earned signal  
  Trap (with imp_change_pct)                    1.000 Near-perfect (leaked!)
  Base rate (predict majority)                  0.818 Dumb baseline       

What happened: imp_change_pct is the percent change from February impressions to
March impressions. The label is_declining is defined as March being <80% of
February. A model that sees imp_change_pct learns: if imp_change_pct is less
than roughly -0.2, predict declining. That's DIRECT LEAKAGE — the feature
contains the label's computation.

Lesson: Never feed a column derived from the same period you're trying to
predict. trend_pct, impression_change_pct, or any rate that compares the
label window t

---
## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.